<p><font size="6" color='grey'> <b>
KI-Agenten. Planen. Handeln. Prüfen.
</b></font> </br></p>



<p><font size="5" color='grey'> <b>
StateGraph Basics
</b></font> </br></p>

---

**Beitrag zum Leitprojekt:** Der Briefing-State ist der Arbeitszustand des PMO-Assistent innerhalb eines Laufs — **Planen** heißt hier: Analyse, Entwurf und Qualitätsprüfung greifen auf dieselbe, gemeinsam geteilte Wahrheitsquelle zu, statt Ergebnisse nur lose durchzureichen.

**Praxis-Transfer:** Mara Vogt braucht einen stabilen State, damit Frage, gefundene Quellen, Sicherheit und Freigabestatus über den gesamten Briefing-Ablauf erhalten bleiben.


In [ ]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

# LangSmith Env-Vars VOR allen LangChain-Imports setzen
import os
os.environ["LANGSMITH_TRACING"] = "false"
from langsmith import tracing_context
# Tracing bleibt standardmäßig aus; ausgewählte Referenzläufe werden explizit aktiviert.
os.environ["LANGSMITH_PROJECT"]    = "m08_stategraph_basics"
os.environ["LANGSMITH_ENDPOINT"]   = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace,
)

setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# Modell-Konfiguration — Rollen als Konstanten
from genai_lib.model_config import (
    BASELINE,
    ROUTER,
    WORKER,
    CODING,
    JUDGE,
    PLANNER,
    WORKER_PREMIUM,
    FRONTIER,
    IMAGE_GENERATION,
    TRANSCRIPTION,
    TRANSCRIPTION_SEGMENTS,
    EMBEDDINGS,
)

# LangSmith Tracing
run_cfg = {
    "tags": ["stategraph"],
    "metadata": {"notebook": "stategraph_basics", "version": "1.0"}
}

**Was passiert hier?**

1. `StateGraph(...)` — definiert den Graphen mit dem State-Schema

# 1 | Übersicht
---

<p><font color='darkblue' size="4">
 <b>Viz</b>
</font></p>

- [LangGraph](https://ralf-42.github.io/Agenten/visualisierungen/langgraph_explainer/)
- [KI-Agenten-Architektur](https://ralf-42.github.io/Agenten/visualisierungen/ki_agent_architektur/)




*Warum LangGraph?* hat gezeigt, **warum** LangGraph existiert und einen ersten 1-Node-Graphen gebaut.  
Dieses Modul geht tiefer: Wir bauen einen **vollständigen 2-Node-Graphen** mit mehreren State-Feldern,  
verschiedenen Edge-Typen und vollständiger Visualisierung.



**Was behandelt dieses Modul**

| Thema | Inhalt |
|-------|--------|
| **State-Design** | TypedDict mit mehreren Feldern, Reducer vs. Überschreiben |
| **Node-Regeln** | Signatur, partieller Return, Fehlerbehandlung |
| **Edges** | `add_edge()`, kurze Vorschau auf `add_conditional_edges()` |
| **Kompilierung** | `compile()`, `recursion_limit`, Graph-Visualisierung |
| **Ausführung** | `invoke()`, `stream()`, State nach Ausführung inspizieren |
| **LangSmith** | Trace mit `run_name`, Node-by-Node-Sicht |


In [ ]:
#@markdown   <p><font size="4" color='green'>  StateGraph</font> </br></p>



diagram = '''
%%{init: {
  'theme': 'base',
  'themeVariables': {
    'primaryColor': '#e0f2fe',
    'primaryBorderColor': '#0284c7',
    'primaryTextColor': '#1e293b',
    'lineColor': '#64748b',
    'clusterBkg': '#f8fafc',
    'clusterBorder': '#cbd5e1',
    'noteBkgColor': '#fef3c7',
    'noteBorderColor': '#d97706',
    'noteTextColor': '#78350f',
    'edgeLabelBackground': '#ffffff'
  }
}}%%
flowchart LR
    STATE[/"📦 State\n(zentrale Wahrheit)"/]
    NODE_A["Node A\nliest State\nschreibt State"] --> STATE
    STATE --> NODE_B["Node B\nliest State\nschreibt State"]

    STATE --> EDGE{"Edge /\nRouter"}
    EDGE -->|"Pfad 1"| NODE_A
    EDGE -->|"Pfad 2"| NODE_B

    %% Farbdefinitionen nach Funktionsgruppe
    classDef default fill:#e0f2fe,stroke:#0284c7,stroke-width:2px,color:#0c4a6e;
    classDef metric fill:#fef3c7,stroke:#d97706,stroke-width:2px,color:#78350f;
    classDef target fill:#fae8ff,stroke:#c026d3,stroke-width:2px,color:#701a75;

    %% Zuweisung
    class EDGE metric;
    class STATE target;
'''
mermaid(diagram, width=800)


**Das Praxisbeispiel: Qualitäts-Agent**

Es wird ein **zweistufiger Analyse-Agent** gebaut:

- **Node 1 `entwurf_node`**: Erstellt einen ersten Textentwurf
- **Node 2 `korrektorat_node`**: Verbessert den Entwurf

Jeder Node liest aus dem **gemeinsamen State** und schreibt seine Ergebnisse zurück.

In [ ]:
from langchain.chat_models import init_chat_model
llm = init_chat_model(JUDGE)

In [ ]:
#@markdown   <p><font size="4" color='green'>  StateGraph mit 2 Knoten</font> </br></p>

diagram = '''
%%{init: {
  'theme': 'base',
  'themeVariables': {
    'primaryColor': '#e0f2fe',
    'primaryBorderColor': '#0284c7',
    'primaryTextColor': '#1e293b',
    'lineColor': '#64748b',
    'clusterBkg': '#f8fafc',
    'clusterBorder': '#cbd5e1',
    'noteBkgColor': '#fef3c7',
    'noteBorderColor': '#d97706',
    'noteTextColor': '#78350f',
    'edgeLabelBackground': '#ffffff'
  }
}}%%
flowchart LR
    ST([START])
    N1["<b>Node 1</b>\nentwurf_node()\n▸ liest: query\n▸ schreibt: draft"]
    N2["<b>Node 2</b>\nkorrektorat_node()\n▸ liest: draft\n▸ schreibt: answer"]
    FIN([FINISH])
    S[/"<b>State</b>\nanfrage: str\nentwurf: str\nfinale_antwort: str\nschritt: int"/]

    ST --> N1 --> N2 --> FIN
    N1 <-.->|"lesen / schreiben"| S
    N2 <-.->|"lesen / schreiben"| S

    %% Farbdefinitionen nach Funktionsgruppe
    classDef default fill:#e0f2fe,stroke:#0284c7,stroke-width:2px,color:#0c4a6e;
    classDef input fill:#e0e7ff,stroke:#6366f1,stroke-width:2px,color:#1e1b4b;
    classDef output fill:#dcfce7,stroke:#16a34a,stroke-width:2px,color:#14532d;
    classDef target fill:#fae8ff,stroke:#c026d3,stroke-width:2px,color:#701a75;

    %% Zuweisung
    class ST input;
    class FIN output;
    class S target;
'''

mermaid(diagram, width=900)

# 2 | StateGraph erstellen
---



**Der State: Einzige Wahrheitsquelle**

Der **State** ist das Herzstück jedes LangGraph-Workflows.  
Alle Nodes lesen daraus und schreiben dorthin – sie kommunizieren **nie direkt**.



**TypedDict: Schnell, typsicher, empfohlen**

| Kriterium | TypedDict | Pydantic BaseModel |
|-----------|----------|--------------------|
| **Empfehlung** | ✅ Für Graph-State | ⚠️ Nur für API-Grenzen |
| **Performance** | Kein Overhead | Langsamer (Validation) |
| **Typen** | Statische Hints | Strikte Laufzeit-Validation |



**Reducer: Wie der State aktualisiert wird**

LangGraph unterscheidet zwei Verhaltensweisen bei State-Updates:

| Typ | Verhalten | Beispiel |
|-----|-----------|----------|
| **Überschreiben** (Standard) | Neuer Wert ersetzt alten | `schritt: 2` → `schritt: 3` |
| **Reducer** (`add_messages`) | Neue Werte werden **angehängt** | `[msg1]` + `[msg2]` → `[msg1, msg2]` |

> **Wichtig:** Für `messages` immer `Annotated[list, add_messages]` verwenden –  
> sonst wird der Gesprächsverlauf bei jedem Node-Return überschrieben!

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages

In [ ]:
# ─── State-Definition ────────────────────────────────────────────────────
class QualityState(TypedDict):
    """State des Briefing-Qualitäts-Agenten.

    Alle Felder werden von den Nodes geteilt. Der State bleibt bewusst klein,
    aber die Qualitätsprüfung bezieht sich jetzt auf Quellenbindung und
    Korpusgrenze des PMO-Assistenten.
    """
    messages:       Annotated[list, add_messages]  # Nachrichten immer anhaengen
    query:        str           # Eingabe des Nutzers (unverändert)
    draft:        str           # Node 1 schreibt den Briefing-Entwurf
    sources:        list[str]   # Demo-Quellen/Evidenzen für die Prüfung
    answer: str                 # Node 2 schreibt hier
    quality_status: str         # ok / überarbeiten
    risk:         str           # kurze Begruendung für Review
    step:        int            # Zaehlt Ausführungsschritte

In [ ]:
# ─── Initialer State ─────────────────────────────────────────────────────
start_state: QualityState = {
    "messages":       [],
    "query":        "",    # Wird beim Aufruf gesetzt
    "draft":        "",
    "sources":        [],
    "answer": "",
    "quality_status": "",
    "risk":         "",
    "step":        0,
}

In [ ]:
from langgraph.graph import StateGraph, START, END

# StateGraph mit dem State-Schema initialisieren
agent_builder = StateGraph(QualityState)

# 3 | Nodes definieren
---


**Die 4 Regeln für Node-Funktionen**

Eine Node-Funktion empfängt immer den vollständigen State und gibt ausschließlich ein `dict` zurück — also `def quality_node(state: QualityState) -> dict:`. Dabei müssen nur die veränderten Felder zurückgegeben werden; LangGraph merged automatisch mit dem bestehenden State, sodass unveränderte Felder erhalten bleiben. Treten Fehler auf, werden diese nicht mit `raise` weitergeleitet, sondern als String in ein Fehlerfeld des States geschrieben — das hält den Graphen stabil. Schließlich gilt: Nodes sind pure Funktionen ohne globale Seiteneffekte; der State ist die einzige Kommunikationsebene zwischen Nodes.



**Partieller Return – warum kein vollständiger State?**

LangGraph **merged** automatisch:

```python
**Aktueller State:**
**{ anfrage: "Hallo", entwurf: "", schritt: 0 }**

**Node gibt zurück:**
return {"entwurf": "Erster Entwurf...", "schritt": 1}

**Neuer State nach Merge:**
**{ anfrage: "Hallo", entwurf: "Erster Entwurf...", schritt: 1 }**
```

> `anfrage` bleibt erhalten, obwohl der Node es nicht zurückgegeben hat!

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

# ─── Node 1: Briefing-Entwurf erstellen ──────────────────────────────────
def entwurf_node(state: QualityState) -> dict:
    """Erstellt einen ersten Briefing-Entwurf und übernimmt Quellen aus dem State."""

    print(f"  [Node 1] Briefing-Entwurf erstellen")

    query = state["query"]
    prompt = [
        HumanMessage(content=(
            "Erstelle einen knappen PMO-Briefing-Entwurf. "
            "Nenne nur belegbare Aussagen und markiere fehlende Evidenz.\n\n"
            f"Frage: {query}"
        ))
    ]
    response = llm.invoke(prompt, config={**run_cfg, "run_name": "m08_kap3_entwurf_node"})
    sources = state.get("sources", [])
    return {
        "messages": [*prompt, response],
        "draft": response.content,
        "sources": sources,
        "step": state.get("step", 0) + 1,
    }

In [ ]:
# ─── Node 2: Qualitätsprüfung durchführen ────────────────────────────────
def korrektorat_node(state: QualityState) -> dict:
    """Prüft, ob Quellen für den Entwurf aus Node 1 vorliegen.

    Liest: state["draft"], state["sources"]
    Schreibt: answer, quality_status, risk, step, messages
    """
    print(f"  [Node 2] Prüfe Briefing-Entwurf ({len(state['draft'])} Zeichen)")

    sources = state.get("sources", [])
    hat_quellen = bool(sources)
    status = "ok" if hat_quellen else "ueberarbeiten"
    risk = "Quellen im State vorhanden." if hat_quellen else "Keine belegende Quelle im State."
    prompt = [
        HumanMessage(content=(
            "Verbessere den Briefing-Entwurf nur auf Basis der angegebenen Quellen. "
            "Bei fehlender Evidenz schreibe ausdrücklich 'Nicht im Korpus'.\n\n"
            f"Entwurf:\n{state['draft']}\n\nQuellen: {sources}\nQualitätsstatus: {status}\nRisiko: {risk}"
        ))
    ]
    response = llm.invoke(prompt, config={**run_cfg, "run_name": "m08_kap3_korrektorat_node"})

    return {
        "messages": [*prompt, response],
        "answer": response.content,
        "quality_status": status,
        "risk": risk,
        "step": state["step"] + 1,
    }

In [ ]:
#@markdown   <p><font size="4" color='green'>  Prozess</font> </br></p>

diagram = '''
%%{init: {
  'theme': 'base',
  'themeVariables': {
    'primaryColor': '#e0f2fe',
    'primaryBorderColor': '#0284c7',
    'primaryTextColor': '#1e293b',
    'lineColor': '#64748b',
    'clusterBkg': '#f8fafc',
    'clusterBorder': '#cbd5e1',
    'noteBkgColor': '#fef3c7',
    'noteBorderColor': '#d97706',
    'noteTextColor': '#78350f',
    'edgeLabelBackground': '#ffffff'
  }
}}%%
flowchart TB
    subgraph Node1["entwurf_node() – Node 1"]
        N1R["Liest: query"] --> N1L["LLM: Erstellt Entwurf"]
        N1L --> N1W["Schreibt: draft, step"]
    end

    subgraph State["Gemeinsamer State"]
        direction LR
        F1["query: str"]
        F2["draft: str"]
        F3["answer: str"]
        F4["step: int"]
        F5["messages: list"]
    end

    subgraph Node2["korrektorat_node() – Node 2"]
        N2R["Liest: draft"] --> N2L["LLM: Verbessert Text"]
        N2L --> N2W["Schreibt: answer, step"]
    end

    Node1 <-->|merge| State
    Node2 <-->|merge| State

    %% Farbdefinitionen nach Funktionsgruppe
    classDef default fill:#e0f2fe,stroke:#0284c7,stroke-width:2px,color:#0c4a6e;
    classDef input fill:#e0e7ff,stroke:#6366f1,stroke-width:2px,color:#1e1b4b;
    classDef output fill:#dcfce7,stroke:#16a34a,stroke-width:2px,color:#14532d;
    classDef target fill:#fae8ff,stroke:#c026d3,stroke-width:2px,color:#701a75;

    %% Zuweisung
    class N1R,N2R input;
    class N1W,N2W output;
    class F1,F2,F3,F4,F5 target;
'''

mermaid(diagram, width=1050)

# 4 | Edges verbinden
---



**Edge-Typen**

| Typ | Methode | Wann verwenden |
|-----|---------|----------------|
| **Einfache Edge** | `add_edge(from, to)` | Immer dieser Weg |
| **Conditional Edge** | `add_conditional_edges(from, router, path_map)` | Entscheidung zur Laufzeit |

**START und END**

```python
from langgraph.graph import START, END

**START = spezieller Eingangsknoten (kein Node, kein Code)**
**END   = spezieller Ausgangsknoten (Workflow beenden)**
quality_builder.add_edge(START, "mein_node")   # Einstieg
quality_builder.add_edge("mein_node", END)     # Ausstieg
```

**Conditional Edges – kurze Übersicht**

```python
**Routing-Funktion: gibt den Namen des nächsten Nodes zurück**
def quality_router(state: QualityState) -> Literal["nochmal_verbessern", END]:
    if state["Qualität"] == "schlecht":
        return "nochmal_verbessern"   # Schleife
    return END                         # Fertig

quality_builder.add_conditional_edges(
    "korrektorat",    # Von diesem Node
    quality_router,     # Diese Funktion entscheidet
    {"nochmal_verbessern": "nochmal_verbessern", END: END},
)
```



In [ ]:
# Nodes zum Builder hinzufügen
agent_builder.add_node("draft",    entwurf_node)
agent_builder.add_node("korrektorat", korrektorat_node)

# Edges verbinden: START → draft → korrektorat → END
agent_builder.add_edge(START,         "draft")
agent_builder.add_edge("draft",     "korrektorat")
agent_builder.add_edge("korrektorat", END)
print()

# 5 | Graph kompilieren & testen
---



**Kompilierung: Builder → unveränderlicher Graph** (frozen)

Nach `compile()` kann der Graph **nicht mehr verändert** werden.  
Die Kompilierung prüft den Graphen auf Vollständigkeit und erzeugt den ausführbaren Workflow.

| Option | Beschreibung | Standard |
|--------|-------------|----------|
| `checkpointer=` | Persistenz (InMemorySaver/PostgresSaver) | Kein Checkpointing |
| `interrupt_before=` | Nodes bei denen pausiert wird | `[]` |
| `interrupt_after=` | Nodes nach denen pausiert wird | `[]` |

> **Best Practice:** Immer `draw_mermaid_png()` nach `compile()` aufrufen –  
> zeigt den **tatsächlich** kompilierten Graphen.

In [ ]:
# Graph kompilieren
agent = agent_builder.compile()

In [ ]:
# Graph visualisieren
from IPython.display import Image, display
display(Image(agent.get_graph(xray=True).draw_mermaid_png()))

In [ ]:
# State initialisieren
initial_state: QualityState = {
    "messages":       [],
    "query":        "Welche Entscheidung zur Vektordatenbank gilt aktuell: D1 oder D2?",
    "draft":        "",
    "sources":        [],
    "answer": "",
    "quality_status": "",
    "risk":         "",
    "step":        0,
}

In [ ]:
with tracing_context(enabled=True):
    ergebnis = agent.invoke(initial_state, config={**run_cfg, "run_name": "m08_kap5_agent_ergebnis"})

In [ ]:
mprint("## Ergebnis nach invoke()")
mprint(f"**Schritte ausgeführt:** {ergebnis['step']}")
mprint("---")
mprint("### Entwurf (Node 1):")
mprint(ergebnis["draft"])

mprint("---")

mprint("### Finale Antwort (Node 2):")
mprint(ergebnis["answer"])

mprint(f"**Nachrichten im State:** {len(ergebnis['messages'])} Einträge")

In [ ]:
# State nach Ausführung inspizieren
mprint("### === State-Inspektion ===")
for feld, wert in ergebnis.items():
    if feld == "messages":
        print(f"  messages: {len(wert)} Einträge")
        for i, msg in enumerate(wert):
            typ = type(msg).__name__
            inhalt = msg.content[:100].replace("\n", " ") + "..."
            print(f"    [{i}] {typ}: {inhalt}")
    elif isinstance(wert, str) and len(wert) > 60:
        mprint(f"  {feld}: {wert[:100]}...")
    else:
        mprint(f"  {feld}: {repr(wert)}")

In [ ]:
# Graph mit stream() Schritt für Schritt verfolgen
mprint("### === stream() - Node-by-Node ===\n")

stream_state: QualityState = {
    **initial_state,
    "query": "Wie hat sich der Status von Risiko R2 zwischen dem 17.03. und dem 28.04.2026 verändert?",
}

for step, event in enumerate(agent.stream(stream_state, config={**run_cfg, "run_name": "m08_kap5_streamdemo"}, stream_mode="updates")):
    node_name = list(event.keys())[0]
    node_output = event[node_name]

    print(f"--- Schritt {step + 1}: Node '{node_name}' ---")
    for feld, wert in node_output.items():
        if feld == "messages":
            print(f"  messages: +{len(wert)} neue Einträge")
        elif isinstance(wert, str) and len(wert) > 80:
            print(f"  {feld}: {wert[:80]}...")
        else:
            print(f"  {feld}: {repr(wert)}")
    print()

# 6 | Graph im LangSmith
---



LangGraph-Traces in LangSmith zeigen eine **Node-by-Node-Sicht** des Workflows –
ideal zum Verstehen und Debuggen.

**Was LangSmith für StateGraphen zeigt**

| Ebene | Inhalt | Nutzen |
|-------|--------|--------|
| **Graph-Ebene** | Gesamtlaufzeit, Input/Output | Überblick |
| **Node-Ebene** | Ein Eintrag pro Node | Welcher Node lief wie lang? |
| **LLM-Ebene** | Prompt, Response, Tokens | Prompt-Debugging |
| **State-Diff** | State vor/nach jedem Node | Was hat sich verändert? |

**run_name und Tags**

```python
config = {
    "run_name": "m08_kap6_stategraphtrace",
    "tags":     ["zwei-node-agent", "invoke"],
}
agent.invoke(state, config=config)
```

In [ ]:
# Ausführung mit LangSmith-Konfiguration
langsmith_config = {
    "run_name": "m08_kap6_stategraphtrace",
    "tags":     ["zwei-node-agent", "invoke"],
    "metadata": {"modul": "stategraph_basics", "version": "1.0"},
}

In [ ]:
# Initial State
trace_state: QualityState = {
    "messages":       [],
    "query":        "Welche Aufgaben sind laut Follow-up-Tracker noch offen, und wer ist verantwortlich?",
    "draft":        "",
    "sources":        [],
    "answer": "",
    "quality_status": "",
    "risk":         "",
    "step":        0,
}

In [ ]:
with tracing_context(enabled=True):
    trace_result = agent.invoke(trace_state, config=langsmith_config)

In [ ]:
# Sicherstellen, dass langsmith_config existiert
mprint("## LangSmith-Trace")
mprint("---")
mprint("**Projekt:** stategraph-basics")
mprint(f"**Run-Name:** {langsmith_config['run_name']}")
mprint(f"**Schritte:** {trace_result['step']}")

mprint("**Finale Antwort:**")
mprint(f"{trace_result['answer'][:400]}...")

mprint("> Trace in LangSmith unter Projekt **stategraph-basics** einsehen.")

**Was passiert hier?**

1. `StateGraph(...)` — definiert den Graphen mit dem State-Schema

In [ ]:
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("m08_stategraph_basics", limit=3, show_steps=True)

# A | Aufgaben
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabenstellungen unten bieten Anregungen; alternative Herausforderungen sind möglich.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs darf und soll generative KI auch als Unterstützung beim Lernen und Entwickeln genutzt werden. Geeignet ist sie zum Beispiel, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.


<p><font color='black' size="5">
Drei-Node-Briefing-Graph
</font></p>

Einen StateGraph für den PMO-Assistenten bauen. Der Graph analysiert eine Frage, erstellt einen Antwortentwurf und prüft die Qualität.


**Grundlagen**
1. Einen einfachen State mit den Feldern `frage`, `analyse`, `antwort` und `status` definieren.
2. Einen linearen Graphen mit mindestens 2 Nodes bauen.
3. Den Graph einmal mit `invoke()` testen.


In [ ]:
# Grundlagen: StateGraph mit mindestens 2 Nodes
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class BriefingState(TypedDict):
    query: str
    analyse: str
    answer: str
    status: str

# 1. frage_analysieren(): liest 'query', schreibt eine kurze Einschätzung nach 'analyse'
# 2. antwort_entwerfen(): liest 'analyse', schreibt einen Entwurf nach 'answer'
# 3. Linearen Graph (mindestens 2 Nodes) aufbauen, kompilieren (my_agent) und mit invoke() testen (grundlagen_ergebnis)

**Aufbau**
1. Einen dritten Node `qualitaets_check` ergänzen.
2. Nodes sequentiell verbinden: START -> Analyse -> Entwurf -> Qualität -> END.
3. Den Graph kompilieren und optional mit `draw_mermaid_png()` visualisieren.
4. Einen Test mit einer Briefing-Frage durchführen.


In [ ]:
# Aufbau: 3-Node-Briefing-Graph
# Startpunkt: my_agent aus Grundlagen um qualitaets_check erweitern

# 1. qualitaets_check(): liest 'analyse'/'answer', ergänzt einen Qualitätshinweis
# 2. Graph sequentiell verbinden: START -> frage_analysieren -> antwort_entwerfen -> qualitaets_check -> END
# 3. Kompilieren (my_agent), optional visualisieren, mit einer Testfrage aufrufen (aufbau_ergebnis)

**Vertiefung**
1. `stream()` statt `invoke()` verwenden und den State nach jedem Node einzeln ausgeben.
2. In einer Zeile dokumentieren, was sich am Ausgabe-Verhalten ändert.
3. LangSmith-Tracing aktiv lassen und prüfen, welche Nodes im Trace sichtbar sind.
4. Optional ein Feld `risiko: str` ergänzen und im Qualitätsnode setzen.


In [ ]:
# Vertiefung: stream() statt invoke()
# Startpunkt: my_agent aus Aufbau

# 1. my_agent.stream(state, stream_mode="updates") iterieren, jedes Update in stream_ergebnisse sammeln
# 2. In stream_beobachtung (>= 30 Zeichen) den Unterschied zu invoke() beschreiben
# 3. Optional: Feld 'risk' im State ergänzen und im Qualitätsnode setzen

# B | Dokumente zum Weiterlesen
---

Ergänzende Artikel aus der Kurs-Dokumentation:

- [State Management](https://ralf-42.github.io/Agenten/04-agenten-implementierung/ablauf-zustand/state-management.html)
- [Einsteiger LangGraph](https://ralf-42.github.io/Agenten/05-frameworks/einsteiger-langgraph.html)
- [LangGraph Best Practices](https://ralf-42.github.io/Agenten/05-frameworks/langgraph-best-practices.html)
- [LangChain/LangGraph Cheatsheet](https://ralf-42.github.io/Agenten/05-frameworks/langchain-langgraph-cheatsheet.html)
- [Checkliste Agentensystem](https://ralf-42.github.io/Agenten/04-agenten-implementierung/checkliste-agentensystem.html)
- [Code Standards](https://ralf-42.github.io/Agenten/10-ressourcen/standards.html)

